In [25]:
import numpy as np
import pandas as pd
import plotly.express as px
from ydata_profiling import ProfileReport

from sklearn.model_selection import train_test_split , GridSearchCV 
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler, OneHotEncoder, OrdinalEncoder 
from sklearn.impute import SimpleImputer

# Building the data pipelines

In [26]:
# load processed data
XY_Data= pd.read_csv(r'D:\K_REPO\ITI\LEC 2 task\XY_Data.csv')
XY_Data.head()
pd.set_option('display.max_columns', None)

In [27]:
# label features 
Numerical_features= ['Fireplaces', 'OverallQual', 'OverallCond', 'PoolArea', 'MasVnrArea', 'KitchenAbvGr', 'WoodDeckSF', 'SalePrice', 'MiscVal', 'GrLivArea', 'BsmtFinSF1', 'LowQualFinSF', 'ScreenPorch', 'GarageCars', '2ndFlrSF', 'BedroomAbvGr', 'HalfBath', 'BsmtFinSF2', 'EnclosedPorch', 'OpenPorchSF', 'TotalBsmtSF', 'FullBath', 'LotArea', 'BsmtFullBath', 'house_Age', 'BsmtUnfSF', 'BsmtHalfBath', 'LotFrontage', '3SsnPorch'] 


Nominal_features= ['MSSubClass', 'MSZoning', 'LandContour', 'LotConfig', 'Neighborhood', 'Condition1', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'Foundation', 'Heating', 'CentralAir', 'GarageType', 'MiscFeature', 'SaleType', 'SaleCondition'] 


Ordinal_features=  ['LotShape', 'LandSlope', 'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'HeatingQC', 'Electrical', 'KitchenQual', 'Functional', 'FireplaceQu', 'GarageFinish', 'GarageQual', 'PavedDrive', 'PoolQC', 'Fence']

In [28]:
XY_Data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1452 entries, 0 to 1451
Data columns (total 68 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   OpenPorchSF    1452 non-null   int64  
 1   ScreenPorch    1452 non-null   int64  
 2   MasVnrArea     1452 non-null   float64
 3   LotFrontage    1452 non-null   float64
 4   BsmtFinSF1     1452 non-null   int64  
 5   GarageCars     1452 non-null   int64  
 6   TotalBsmtSF    1452 non-null   int64  
 7   HalfBath       1452 non-null   int64  
 8   OverallQual    1452 non-null   int64  
 9   MiscVal        1452 non-null   int64  
 10  LowQualFinSF   1452 non-null   int64  
 11  BsmtHalfBath   1452 non-null   int64  
 12  OverallCond    1452 non-null   int64  
 13  FullBath       1452 non-null   int64  
 14  3SsnPorch      1452 non-null   int64  
 15  GrLivArea      1452 non-null   int64  
 16  Fireplaces     1452 non-null   int64  
 17  BsmtFinSF2     1452 non-null   int64  
 18  KitchenA

## I will make 3 sub-Pipelines for eache tybe of features 

### Numeric Pipeline (Logscale to handle outliers + Standrdization)

In [29]:

Features_to_Log_and_Standrize = [ 'WoodDeckSF', 'LowQualFinSF', 'BsmtUnfSF', 'BsmtFinSF1', '2ndFlrSF', 'PoolArea', 'OpenPorchSF', 'GrLivArea', 'MasVnrArea', '3SsnPorch', 'house_Age', 'LotArea', 'TotalBsmtSF', 'ScreenPorch', 'LotFrontage', 'BsmtFinSF2', 'EnclosedPorch', 'MiscVal']
Features_to_Standrize_only = [ 'OverallCond', 'Fireplaces', 'OverallQual', 'BedroomAbvGr' , 'GarageCars', 'FullBath', 'BsmtHalfBath', 'HalfBath', 'KitchenAbvGr',  'BsmtFullBath']


X = XY_Data.drop('SalePrice', axis=1)
y= XY_Data['SalePrice'].values.reshape(-1,1)
log_y=np.log1p(y)

In [30]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

##### log + Standardization Sub-Pipeline

In [31]:
log_standrize_Pipe = Pipeline([ ('Log_scale', FunctionTransformer(np.log1p)),
                                ('Standrize', StandardScaler())
                                 ])

##### Standardization Only Sub-Pipeline

In [32]:
Standrize_Pipe = Pipeline([  ('Standrize', StandardScaler())  ])

In [33]:
transformers=[ ('Log & Standrize', log_standrize_Pipe , Features_to_Log_and_Standrize),
                ('Standrize only', Standrize_Pipe, Features_to_Standrize_only),
                ]
Processor= ColumnTransformer(transformers, remainder='passthrough')
    

In [34]:
Processor

,transformers,"[('Log & Standrize', ...), ('Standrize only', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,func,<ufunc 'log1p'>
,inverse_func,None
,validate,False


##### make sure of transformed Numerical data

In [35]:
explore_Numerical_trans = Processor.fit_transform(X_train, y_train)
print(explore_Numerical_trans.shape)
type(explore_Numerical_trans)

(1161, 67)


numpy.ndarray

In [36]:
Numerical_transfomed_data = pd.DataFrame(explore_Numerical_trans)
Numerical_transfomed_data.head(10)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66
0,-0.928208,-0.137808,0.189927,0.648416,1.085432,-0.077865,-1.08143,0.172601,1.173504,-0.134828,0.651529,0.41671,0.024136,-0.29405,0.4968,-0.346227,-0.398023,-0.194542,-0.500911,-0.957524,-0.78271,0.144923,-1.024711,-1.033004,-0.239988,1.222206,-0.217762,-0.801666,50,RL,Lvl,Inside,NAmes,Feedr,1Fam,1.5Fin,Gable,CompShg,MetalSd,MetalSd,BrkFace,CBlock,GasW,Y,Attchd,NaN,WD,Normal,Reg,Gtl,TA,TA,TA,TA,No,BLQ,Unf,TA,FuseA,TA,Typ,NaN,Unf,TA,Y,NaN,GdWo
1,-0.928208,-0.137808,-0.4664,0.919702,-0.879748,-0.077865,0.734609,-0.637646,-0.808002,-0.134828,0.776782,0.338101,0.268923,-0.29405,0.069711,-0.346227,-0.398023,-0.194542,-0.500911,-0.957524,-0.78271,0.144923,-1.024711,-1.033004,-0.239988,-0.76795,-0.217762,1.113425,190,RL,Lvl,Inside,Edwards,Norm,2fmCon,1Story,Gable,CompShg,MetalSd,MetalSd,NaN,CBlock,GasW,N,Detchd,NaN,ConLD,Normal,Reg,Gtl,TA,TA,TA,TA,No,ALQ,Unf,TA,SBrkr,TA,Typ,NaN,Unf,TA,Y,NaN,NaN
2,-0.928208,-0.137808,0.436183,-1.394339,1.097311,-0.077865,0.753277,0.250509,1.050272,-0.134828,0.289774,-0.148827,-0.228743,3.50245,0.159878,-0.346227,-0.398023,-0.194542,0.399333,0.567422,-0.058625,0.144923,0.310518,0.781395,-0.239988,1.222206,-0.217762,-0.801666,60,RL,Lvl,Inside,SawyerW,Norm,1Fam,2Story,Gable,CompShg,HdBoard,HdBoard,BrkFace,CBlock,GasA,Y,Attchd,NaN,WD,Normal,Reg,Gtl,TA,TA,Gd,TA,No,Unf,Unf,TA,SBrkr,TA,Typ,TA,Unf,TA,Y,NaN,MnPrv
3,0.99925,-0.137808,-0.359433,0.786574,-0.879748,-0.077865,-1.08143,-1.58975,1.09678,-0.134828,-1.090956,-1.354777,-0.00166,-0.29405,0.159878,-0.346227,-0.398023,-0.194542,-0.500911,0.567422,-0.058625,-2.276024,0.310518,-1.033004,-0.239988,-0.76795,-0.217762,1.113425,120,RM,Lvl,Inside,CollgCr,Norm,TwnhsE,1Story,Gable,CompShg,VinylSd,VinylSd,BrkFace,PConc,GasA,Y,Attchd,NaN,WD,Normal,Reg,Gtl,Gd,TA,Gd,TA,Av,GLQ,Unf,Ex,SBrkr,Gd,Typ,TA,RFn,TA,Y,NaN,NaN
4,-0.928208,-0.137808,0.071162,0.557622,-0.879748,-0.077865,-1.08143,-1.397547,-0.808002,-0.134828,1.079331,0.183109,-0.186457,-0.29405,0.159878,-0.346227,-0.398023,-0.194542,-4.101889,-0.957524,-3.679047,-2.276024,-2.35994,-2.847403,-0.239988,1.222206,-0.217762,1.113425,30,RL,Low,Inside,Edwards,Norm,1Fam,1Story,Gable,CompShg,Wd Sdng,Wd Sdng,NaN,BrkTil,GasA,N,NaN,NaN,WD,Normal,IR1,Sev,Fa,Fa,Fa,Po,Gd,BLQ,Unf,Gd,FuseA,Fa,Maj1,NaN,NaN,NaN,Y,NaN,NaN
5,0.847106,-0.137808,0.464028,-1.394339,1.175826,-0.077865,0.628097,0.242808,-0.808002,-0.134828,-1.090956,0.508515,-0.185208,-0.29405,1.893918,-0.346227,-0.398023,-0.194542,-0.500911,0.567422,-0.058625,0.144923,0.310518,0.781395,-0.239988,1.222206,-0.217762,-0.801666,60,RL,Lvl,Inside,Gilbert,Norm,1Fam,2Story,Gable,CompShg,VinylSd,VinylSd,NaN,PConc,GasA,Y,BuiltIn,NaN,WD,Normal,IR2,Gtl,Gd,TA,Gd,TA,Av,Unf,Unf,Ex,SBrkr,Gd,Typ,Gd,Fin,TA,Y,NaN,NaN
6,-0.928208,-0.137808,-0.428372,0.828232,1.316183,12.646948,1.12451,1.984791,1.181836,-0.134828,0.239306,1.143734,0.42177,4.048804,0.575695,3.074072,-0.398023,5.794821,-0.500911,2.092368,0.665459,2.56587,0.310518,2.595794,-0.239988,1.222206,-0.217762,1.113425,60,RL,Lvl,Inside,NWAmes,RRAn,1Fam,2Story,Gable,CompShg,Plywood,Plywood,BrkFace,CBlock,GasA,Y,Attchd,TenC,WD,Normal,IR1,Gtl,TA,TA,Gd,TA,No,BLQ,LwQ,TA,SBrkr,Gd,Typ,TA,RFn,TA,Y,Fa,MnPrv
7,0.827575,-0.137808,-0.652188,0.794589,1.111748,-0.077865,0.939419,0.146841,-0.808002,-0.134828,-0.566416,0.393839,-0.049295,-0.29405,0.159878,-0.346227,-0.398023,-0.194542,-0.500911,0.567422,-0.058625,0.144923,0.310518,0.781395,-0.239988,1.222206,-0.217762,1.113425,60,RL,HLS,FR2,Gilbert,Norm,1Fam,2Story,Gable,CompShg,VinylSd,VinylSd,NaN,PConc,GasA,Y,Attchd,NaN,WD,Normal,IR1,Gtl,TA,TA,Gd,TA,Av,GLQ,Unf,Gd,SBrkr,TA,Typ,TA,Fin,TA,Y,NaN,NaN
8,1.186406,-0.137808,-0.228815,0.412465,1.134777,-0.077865,-1.08143,0.471028,0.919684,-0.134828,0.3138,0.389972,0.066091,-0.29405,0.159878,3.10958,-0.398023,-0.194542,-0.500911,0.567422,-0.058625,0.144923,0.310518,0.78139

### Nominal Pipeline (Imputation + One Hot Encoder )

In [37]:
Nominal_Pipeline= Pipeline(steps= [ ('imputation with none', SimpleImputer(strategy='constant', fill_value='None')),  
                                ('encoding', OneHotEncoder(handle_unknown='ignore'))    ]   )

In [38]:
XY_Data[Nominal_features].isna().sum()[lambda x : x>0]

MasVnrType      864
GarageType       81
MiscFeature    1398
dtype: int64

In [39]:
Nominal_transformer= ColumnTransformer(transformers=[('Nominal Imputer & encoder' , Nominal_Pipeline, Nominal_features)] , remainder='passthrough')

In [40]:
Nominal_transformer

,transformers,"[('Nominal Imputer & encoder', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'constant'
,fill_value,'None'


##### make sure of transformed Nominal data

In [41]:

explore_Nominal_pipeline = Nominal_transformer.fit_transform(X_train , y_train)

In [42]:
explore_Nominal_pipeline.shape

(1161, 209)

In [43]:
Nominal_transformed_data =pd.DataFrame(explore_Nominal_pipeline)
Nominal_transformed_data.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0,0,180.0,78.0,460,1,874,1,5,0,0,0,5,1,0,1524,0,0,1,0,3,49,650,414,0,0,0,11344,Reg,Gtl,TA,TA,TA,TA,No,BLQ,Unf,TA,FuseA,TA,Typ,NaN,Unf,TA,Y,NaN,GdWo
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,48,0,0.0,68.0,1040,1,1164,0,5,0,0,0,5,1,0,1164,0,0,1,0,3,58,0,124,1,0,0,10880,Reg,Gtl,TA,TA,TA,TA,No,ALQ,Unf,TA,SBrkr,TA,Typ,NaN,Unf,TA,Y,NaN,NaN
2,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,50,204,130.0,70.0,0,2,650,1,6,0,0,0,6,2,0,1564,1,0,1,0,3,30,676,650,0,0,0,8400,Reg,Gtl,TA,TA,Gd,TA,No,Unf,Unf,TA,SBrkr,TA,Typ,TA,Unf,TA,Y,NaN,MnPrv
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0,0,147.0,70.0,697,2,848,0,6,0,0,0,5,1,0,848,1,0,1,149,1,4,0,151,1,0,0,4426,Reg,Gtl,Gd,TA,Gd,TA,Av,GLQ,Unf,Ex,SBrkr,Gd,Typ,TA,RFn,TA,Y,NaN,NaN
4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.

### Ordinal data Pipeline (Imputation + OrdinalEncoder)

In [44]:
XY_Data[Ordinal_features].isna().sum()[lambda x : x>0]

BsmtQual          37
BsmtCond          37
BsmtExposure      38
BsmtFinType1      37
BsmtFinType2      38
FireplaceQu      686
GarageFinish      81
GarageQual        81
PoolQC          1445
Fence           1171
dtype: int64

###### Maps of ordinal encoding

In [45]:
Ordinal_features=  ['LotShape', 'LandSlope', 'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1',
     'BsmtFinType2', 'HeatingQC', 'Electrical', 'KitchenQual', 'Functional', 'FireplaceQu', 'GarageFinish', 'GarageQual', 'PavedDrive', 'PoolQC', 'Fence']
ordinal_map = {
    'LotShape':     ['Nane','Reg','IR1','IR2','IR3'],
    'LandSlope':    ['Nane','Sev','Mod','Gtl'],
    'ExterQual':    ['Nane','Po','Fa','TA','Gd','Ex'],
    'ExterCond':    ['Nane','Po','Fa','TA','Gd','Ex'],
    'BsmtQual':     ['Nane','Po','Fa','TA','Gd','Ex'],
    'BsmtCond':     ['Nane','Po','Fa','TA','Gd','Ex'],
    'BsmtExposure': ['Nane','No','Mn','Av','Gd'],
    'BsmtFinType1': ['Nane','Unf','LwQ','Rec','BLQ','ALQ','GLQ'],
    'BsmtFinType2': ['Nane','Unf','LwQ','Rec','BLQ','ALQ','GLQ'],
    'HeatingQC':    ['Nane','Po','Fa','TA','Gd','Ex'],
    'Electrical':   ['Nane','Mix','FuseP','FuseF','FuseA','SBrkr'],
    'KitchenQual':  ['Nane','Po','Fa','TA','Gd','Ex'],
    'Functional':   ['Nane','Sal','Sev','Maj2','Maj1','Mod','Min2','Min1','Typ'],
    'FireplaceQu':  ['Nane','Po','Fa','TA','Gd','Ex'],
    'GarageFinish': ['Nane','Unf','RFn','Fin'],
    'GarageQual':   ['Nane','Po','Fa','TA','Gd','Ex'],
    'PavedDrive':   ['Nane','N','P','Y'],
    'PoolQC':       ['Nane','Fa','TA','Gd','Ex'],
    'Fence':        ['Nane','MnWw','GdWo','MnPrv','GdPrv']
}


In [46]:
Ordinal_Pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Nane')),
    ('ordinal_encoder', OrdinalEncoder(
        categories=[ordinal_map[col] for col in Ordinal_features]
    ))
])

In [47]:
Ordinal_transformer= ColumnTransformer(transformers=[
    ('Ordinal data Imputaion & Encoding',   Ordinal_Pipeline,   Ordinal_features)
], remainder='passthrough')

In [51]:
Explore_Ordinal_transformed_data = Ordinal_transformer.fit_transform(X_train,y_train)

In [52]:
Explore_Ordinal_transformed_data.shape

(1161, 67)

In [ ]:
Explore_Ordinal_transformed_df= pd.DataFrame(Explore_Ordinal_transformed_data)
Explore_Ordinal_transformed_df.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66
0,1.0,3.0,3.0,3.0,3.0,3.0,1.0,4.0,1.0,3.0,4.0,3.0,8.0,0.0,1.0,3.0,3.0,0.0,2.0,0,0,180.0,78.0,460,1,874,1,5,0,0,0,5,1,0,1524,0,0,1,0,3,49,650,414,0,0,0,11344,50,RL,Lvl,Inside,NAmes,Feedr,1Fam,1.5Fin,Gable,CompShg,MetalSd,MetalSd,BrkFace,CBlock,GasW,Y,Attchd,NaN,WD,Normal
1,1.0,3.0,3.0,3.0,3.0,3.0,1.0,5.0,1.0,3.0,5.0,3.0,8.0,0.0,1.0,3.0,3.0,0.0,0.0,48,0,0.0,68.0,1040,1,1164,0,5,0,0,0,5,1,0,1164,0,0,1,0,3,58,0,124,1,0,0,10880,190,RL,Lvl,Inside,Edwards,Norm,2fmCon,1Story,Gable,CompShg,MetalSd,MetalSd,NaN,CBlock,GasW,N,Detchd,NaN,ConLD,Normal
2,1.0,3.0,3.0,3.0,4.0,3.0,1.0,1.0,1.0,3.0,5.0,3.0,8.0,3.0,1.0,3.0,3.0,0.0,3.0,50,204,130.0,70.0,0,2,650,1,6,0,0,0,6,2,0,1564,1,0,1,0,3,30,676,650,0,0,0,8400,60,RL,Lvl,Inside,SawyerW,Norm,1Fam,2Story,Gable,CompShg,HdBoard,HdBoard,BrkFace,CBlock,GasA,Y,Attchd,NaN,WD,Normal
3,1.0,3.0,4.0,3.0,4.0,3.0,3.0,6.0,1.0,5.0,5.0,4.0,8.0,3.0,2.0,3.0,3.0,0.0,0.0,0,0,147.0,70.0,697,2,848,0,6,0,0,0,5,1,0,848,1,0,1,149,1,4,0,151,1,0,0,4426,120,RM,Lvl,Inside,CollgCr,Norm,TwnhsE,1Story,Gable,CompShg,VinylSd,VinylSd,BrkFace,PConc,GasA,Y,Attchd,NaN,WD,Normal
4,2.0,1.0,2.0,2.0,2.0,1.0,4.0,4.0,1.0,4.0,4.0,2.0,4.0,0.0,0.0,0.0,3.0,0.0,0.0,0,0,0.0,70.0,350,0,683,1,1,0,0,0,1,0,0,904,0,0,1,0,1,87,0,333,1,0,0,10020,30,RL,Low,Inside,Edwards,Norm,1Fam,1Story,Gable,CompShg,Wd Sdng,Wd Sdng,NaN,BrkTil,GasA,N,NaN,NaN,WD,Normal
